# 00 Paper Figure Storyboard

This notebook is a reading-oriented storyboard for the RouteRec paper figures.

It is intentionally more opinionated than the section-local notebooks under `01_` to `A05_`.
The goal is not only to render plots, but to answer three questions while reading top-to-bottom:

- Which figures are worth putting into the main paper?
- What evidence does each figure provide beyond a plain accuracy bar plot?
- Which artifacts already exist in `writing/results` or `experiments/run/artifacts/results/fmoe_n4/diag`, and which parts are still demo placeholders?

The notebook prefers actual local CSV/diagnostic artifacts when available, and falls back to demo data otherwise.


In [ ]:
from pathlib import Path
import sys
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

RESULTS_ROOT = Path("/workspace/FeaturedMoE/writing/results")
DIAG_ROOT = Path("/workspace/FeaturedMoE/experiments/run/artifacts/results/fmoe_n4/diag")
if str(RESULTS_ROOT) not in sys.path:
    sys.path.insert(0, str(RESULTS_ROOT))

plt.style.use("default")
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,
})

GENERATED = RESULTS_ROOT / "generated_figures"
GENERATED.mkdir(parents=True, exist_ok=True)

def story_note(title: str, body: str) -> None:
    display(Markdown(f"### {title}\n\n{body}"))

def load_csv_or_demo(csv_path, required_columns, demo_builder=None):
    csv_path = Path(csv_path)
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        missing = [column for column in required_columns if column not in df.columns]
        if missing:
            raise ValueError(f"{csv_path.name} is missing required columns: {missing}")
        if not df.empty:
            return df, "csv"
    if demo_builder is None:
        return pd.DataFrame(columns=required_columns), "empty"
    demo_df = demo_builder()
    return demo_df, "demo"

def export_figure(fig, output_stem, results_root, formats=("png", "pdf"), dpi=300):
    output_dir = Path(results_root) / "generated_figures"
    output_dir.mkdir(parents=True, exist_ok=True)
    saved_paths = []
    for extension in formats:
        output_path = output_dir / f"{output_stem}.{extension}"
        fig.savefig(output_path, dpi=dpi, bbox_inches="tight")
        saved_paths.append(output_path)
    return saved_paths

def _color_map(labels):
    cmap = plt.get_cmap("tab10")
    return {label: cmap(i % 10) for i, label in enumerate(labels)}

def grouped_barplot(data, x, hue, y, ax, title=None, ylabel=None, xlabel=None, rotate=0):
    data = data.copy()
    x_order = list(dict.fromkeys(data[x].tolist()))
    hue_order = list(dict.fromkeys(data[hue].tolist()))
    colors = _color_map(hue_order)
    width = 0.8 / max(len(hue_order), 1)
    positions = np.arange(len(x_order))
    for idx, hue_value in enumerate(hue_order):
        subset = data[data[hue] == hue_value]
        values = []
        for x_value in x_order:
            match = subset[subset[x] == x_value]
            values.append(float(match.iloc[0][y]) if not match.empty else np.nan)
        offsets = positions - 0.4 + width / 2 + idx * width
        bars = ax.bar(offsets, values, width=width, label=str(hue_value), color=colors[hue_value])
        for bar in bars:
            height = bar.get_height()
            if not np.isnan(height):
                ax.annotate(f"{height:.3f}", (bar.get_x() + bar.get_width() / 2, height), textcoords="offset points", xytext=(0, 3), ha="center", fontsize=7)
    ax.set_xticks(positions)
    ax.set_xticklabels(x_order, rotation=rotate)
    if title:
        ax.set_title(title)
    if ylabel:
        ax.set_ylabel(ylabel)
    if xlabel:
        ax.set_xlabel(xlabel)
    ax.legend(frameon=False)

def lineplot_with_markers(data, x, y, hue, ax, title=None, ylabel=None, xlabel=None):
    data = data.copy()
    hue_order = list(dict.fromkeys(data[hue].tolist()))
    colors = _color_map(hue_order)
    for hue_value in hue_order:
        subset = data[data[hue] == hue_value]
        ax.plot(subset[x].tolist(), subset[y].tolist(), marker="o", label=str(hue_value), color=colors[hue_value])
    if title:
        ax.set_title(title)
    if ylabel:
        ax.set_ylabel(ylabel)
    if xlabel:
        ax.set_xlabel(xlabel)
    ax.legend(frameon=False)

def scatterplot_with_annotations(data, x, y, ax, hue=None, style=None, annotate_column=None, title=None, ylabel=None, xlabel=None):
    data = data.copy()
    labels = list(dict.fromkeys(data[hue].tolist())) if hue else ["series"]
    colors = _color_map(labels)
    markers = ["o", "s", "^", "D", "P", "X"]
    style_order = list(dict.fromkeys(data[style].tolist())) if style else [None]
    style_map = {value: markers[i % len(markers)] for i, value in enumerate(style_order)}
    for _, row in data.iterrows():
        color = colors[row[hue]] if hue else "#4c72b0"
        marker = style_map[row[style]] if style else "o"
        ax.scatter(row[x], row[y], color=color, marker=marker, s=60)
        if annotate_column is not None:
            ax.annotate(str(row[annotate_column]), (row[x], row[y]), textcoords="offset points", xytext=(4, 4), fontsize=7)
    if title:
        ax.set_title(title)
    if ylabel:
        ax.set_ylabel(ylabel)
    if xlabel:
        ax.set_xlabel(xlabel)
    if hue:
        from matplotlib.lines import Line2D
        handles = [Line2D([0], [0], marker='o', color='w', label=str(label), markerfacecolor=colors[label], markersize=7) for label in labels]
        ax.legend(handles=handles, frameon=False, title=hue)

def heatmap_from_long(data, index, columns, values, ax, title=None, cmap="viridis", fmt=".3f"):
    pivot = data.pivot(index=index, columns=columns, values=values)
    matrix = pivot.to_numpy(dtype=float)
    im = ax.imshow(matrix, aspect="auto", cmap=cmap)
    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=35, ha="right")
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix[i, j]
            if not np.isnan(value):
                ax.text(j, i, format(value, fmt), ha="center", va="center", fontsize=7, color="white")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    if title:
        ax.set_title(title)


## Recommended Figure Set

A compact and persuasive paper story usually looks stronger when the figures answer different kinds of reviewer questions.

- `Main result`: does RouteRec improve ranking quality?
- `Routing control`: is the gain tied to behavior-guided routing rather than a generic MoE or bigger router?
- `Stage structure`: is the macro-mid-micro architecture chosen for a reason?
- `Cue sanity`: are compact cues enough, and does alignment actually matter?
- `Diagnostics / appendix`: does routing stay interpretable and non-collapsed?
- `PCA / geometry`: can we show an intuitive qualitative picture without overselling it as main evidence?

The table below is the suggested role split.


In [ ]:
plan_df = pd.DataFrame(
    [
        ["Main paper", "Routing control", "Quality + consistency", "Shows the route is actually controlled by behavior cues, not just extra capacity"],
        ["Main paper", "Stage structure", "Ablation + order/layout comparison", "Shows why macro-mid-micro decomposition was chosen"],
        ["Main paper", "Cue sanity", "Compactness + perturbation drop", "Shows lightweight cues are enough, but alignment still matters"],
        ["Appendix", "Routing diagnostics", "Usage / entropy / family consistency", "Shows interpretability and anti-collapse behavior"],
        ["Appendix", "PCA view", "Feature or router-input geometry", "Useful as qualitative support, but weaker than quantitative sanity plots"],
    ],
    columns=["Placement", "Figure", "Primary message", "Why it helps"],
)
plan_df


## 1. Routing Control: Quality Plus Consistency

This is one of the most important main-paper figures.

A pure performance bar plot can show `behavior-guided > shared FFN`, but it still leaves a reviewer wondering whether the gain is just due to another conditional module.
A second panel with a consistency curve answers a stronger question: **when two prefixes are behaviorally similar, does the router behave more similarly too?**

That second panel is what turns a "better model" figure into a "better routing mechanism" figure.


In [ ]:
QUALITY_PATH = RESULTS_ROOT / "02_routing_control/02a_routing_control_quality.csv"
CONSISTENCY_PATH = RESULTS_ROOT / "02_routing_control/02b_routing_control_consistency.csv"
QUALITY_COLUMNS = ["paper_section", "panel", "dataset", "variant_or_model", "metric", "cutoff", "value", "split", "selection_rule", "run_id", "source_path", "notes"]
CONSISTENCY_COLUMNS = ["paper_section", "panel", "dataset", "variant_or_model", "similarity_bucket", "consistency_value", "split", "selection_rule", "run_id", "source_path", "notes"]

def demo_quality() -> pd.DataFrame:
    rows = []
    datasets = ["Beauty", "KuaiRec"]
    values = {
        "shared_ffn": [0.0446, 0.2897],
        "hidden_only": [0.0798, 0.3388],
        "mixed": [0.0801, 0.3391],
        "behavior_only": [0.0806, 0.3398],
    }
    for variant, scores in values.items():
        for dataset, value in zip(datasets, scores):
            rows.append({
                "paper_section": "02_routing_control",
                "panel": "quality",
                "dataset": dataset,
                "variant_or_model": variant,
                "metric": "MRR",
                "cutoff": 20,
                "value": value,
                "split": "test",
                "selection_rule": "storyboard_demo",
                "run_id": "routing_demo",
                "source_path": "demo",
                "notes": "Demo values roughly follow observed routing-control outcomes.",
            })
    return pd.DataFrame(rows)

def demo_consistency() -> pd.DataFrame:
    rows = []
    buckets = ["0.0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0"]
    curves = {
        "hidden_only": [0.33, 0.39, 0.45, 0.50, 0.55],
        "mixed": [0.35, 0.42, 0.49, 0.56, 0.61],
        "behavior_only": [0.39, 0.47, 0.56, 0.64, 0.72],
    }
    for variant, values in curves.items():
        for bucket, value in zip(buckets, values):
            rows.append({
                "paper_section": "02_routing_control",
                "panel": "consistency",
                "dataset": "aggregate",
                "variant_or_model": variant,
                "similarity_bucket": bucket,
                "consistency_value": value,
                "split": "test",
                "selection_rule": "storyboard_demo",
                "run_id": "routing_demo",
                "source_path": "demo",
                "notes": "Demo consistency curve.",
            })
    return pd.DataFrame(rows)

quality_df, quality_mode = load_csv_or_demo(QUALITY_PATH, QUALITY_COLUMNS, demo_builder=demo_quality)
consistency_df, consistency_mode = load_csv_or_demo(CONSISTENCY_PATH, CONSISTENCY_COLUMNS, demo_builder=demo_consistency)
display(Markdown(f"**Load mode:** quality={quality_mode}, consistency={consistency_mode}"))

bucket_order = ["0.0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0"]
consistency_plot = consistency_df.copy()
consistency_plot["similarity_bucket"] = pd.Categorical(consistency_plot["similarity_bucket"], categories=bucket_order, ordered=True)
consistency_plot = consistency_plot.sort_values("similarity_bucket")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), constrained_layout=True)
grouped_barplot(
    quality_df,
    x="dataset",
    hue="variant_or_model",
    y="value",
    ax=axes[0],
    title="(a) Ranking quality",
    ylabel="MRR@20",
    xlabel="Dataset",
)
lineplot_with_markers(
    consistency_plot,
    x="similarity_bucket",
    y="consistency_value",
    hue="variant_or_model",
    ax=axes[1],
    title="(b) Routing consistency by similarity bucket",
    ylabel="Consistency score",
    xlabel="Behavior similarity bucket",
)
saved_paths = export_figure(fig, "00_storyboard_routing_control", RESULTS_ROOT)
display(Markdown("Saved figures: " + ", ".join(str(path) for path in saved_paths)))
plt.show()

story_note(
    "How to use this figure in the paper",
    "Use panel (a) to establish that the design matters for ranking quality, then use panel (b) to explain **why**. A strong caption should say that behavior-guided routing is preferred not only because it improves MRR, but because it produces more behavior-consistent routing decisions for similar prefixes."
)


## 2. Stage Structure: Why This Architecture

This figure should answer a design question, not just another performance question.

A clean version is usually:

- `Panel A`: remove macro, remove mid, remove micro, single-stage, dense full.
- `Panel B`: compare the chosen order/layout against a few credible alternatives.

That combination lets you say both:

- staged decomposition matters, and
- the exact coarse-to-fine composition was not arbitrary.


In [ ]:
STAGE_PATH = RESULTS_ROOT / "03_stage_structure/03a_stage_ablation.csv"
DENSE_PATH = RESULTS_ROOT / "03_stage_structure/03b_dense_vs_staged.csv"
WRAPPER_PATH = RESULTS_ROOT / "03_stage_structure/03c_wrapper_order.csv"
STAGE_COLUMNS = ["paper_section", "panel", "dataset", "variant_or_model", "ablation_group", "metric", "cutoff", "value", "split", "selection_rule", "run_id", "source_path", "notes"]
DENSE_COLUMNS = ["paper_section", "panel", "dataset", "variant_or_model", "layout_variant", "stage_count", "metric", "cutoff", "value", "split", "selection_rule", "run_id", "source_path", "notes"]
WRAPPER_COLUMNS = ["paper_section", "panel", "dataset", "variant_or_model", "wrapper_variant", "stage_order", "metric", "cutoff", "value", "split", "selection_rule", "run_id", "source_path", "notes"]

def demo_stage() -> pd.DataFrame:
    rows = []
    values = {
        "full": 0.1185,
        "remove_macro": 0.1152,
        "remove_mid": 0.1144,
        "remove_micro": 0.1129,
        "single_stage": 0.1137,
        "dense_full": 0.1082,
    }
    for group, value in values.items():
        rows.append({
            "paper_section": "03_stage_structure",
            "panel": "stage_ablation",
            "dataset": "KuaiRec",
            "variant_or_model": "RouteRec",
            "ablation_group": group,
            "metric": "MRR",
            "cutoff": 20,
            "value": value,
            "split": "test",
            "selection_rule": "storyboard_demo",
            "run_id": "stage_demo",
            "source_path": "demo",
            "notes": "Demo stage ablation.",
        })
    return pd.DataFrame(rows)

def demo_dense() -> pd.DataFrame:
    rows = []
    values = [("dense_ffn", 0, 0.1082), ("single_stage", 1, 0.1137), ("two_stage", 2, 0.1161), ("three_stage", 3, 0.1185)]
    for layout_variant, stage_count, value in values:
        rows.append({
            "paper_section": "03_stage_structure",
            "panel": "dense_vs_staged",
            "dataset": "KuaiRec",
            "variant_or_model": "RouteRec",
            "layout_variant": layout_variant,
            "stage_count": stage_count,
            "metric": "MRR",
            "cutoff": 20,
            "value": value,
            "split": "test",
            "selection_rule": "storyboard_demo",
            "run_id": "stage_demo",
            "source_path": "demo",
            "notes": "Demo dense-vs-staged curve.",
        })
    return pd.DataFrame(rows)

def demo_wrapper() -> pd.DataFrame:
    rows = []
    values = [
        ("default", "macro-mid-micro", 0.1185),
        ("order_swap", "mid-macro-micro", 0.1163),
        ("prepend_dense", "dense-macro-mid-micro", 0.1170),
        ("bundle_macro_mid", "macro+mid -> micro", 0.1174),
    ]
    for wrapper_variant, stage_order, value in values:
        rows.append({
            "paper_section": "03_stage_structure",
            "panel": "wrapper_order",
            "dataset": "KuaiRec",
            "variant_or_model": "RouteRec",
            "wrapper_variant": wrapper_variant,
            "stage_order": stage_order,
            "metric": "MRR",
            "cutoff": 20,
            "value": value,
            "split": "test",
            "selection_rule": "storyboard_demo",
            "run_id": "stage_demo",
            "source_path": "demo",
            "notes": "Demo wrapper/order comparison.",
        })
    return pd.DataFrame(rows)

stage_df, stage_mode = load_csv_or_demo(STAGE_PATH, STAGE_COLUMNS, demo_builder=demo_stage)
dense_df, dense_mode = load_csv_or_demo(DENSE_PATH, DENSE_COLUMNS, demo_builder=demo_dense)
wrapper_df, wrapper_mode = load_csv_or_demo(WRAPPER_PATH, WRAPPER_COLUMNS, demo_builder=demo_wrapper)
display(Markdown(f"**Load mode:** stage={stage_mode}, dense={dense_mode}, wrapper={wrapper_mode}"))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), constrained_layout=True)
grouped_barplot(stage_df, x="ablation_group", hue="dataset", y="value", ax=axes[0], title="(a) Stage ablation", ylabel="MRR@20", xlabel="Variant", rotate=25)
scatterplot_with_annotations(dense_df, x="stage_count", y="value", annotate_column="layout_variant", ax=axes[1], title="(b) Dense vs staged", ylabel="MRR@20", xlabel="Number of routed stages")
grouped_barplot(wrapper_df, x="wrapper_variant", hue="dataset", y="value", ax=axes[2], title="(c) Order and wrapper choices", ylabel="MRR@20", xlabel="Layout variant", rotate=20)
saved_paths = export_figure(fig, "00_storyboard_stage_structure", RESULTS_ROOT)
display(Markdown("Saved figures: " + ", ".join(str(path) for path in saved_paths)))
plt.show()

story_note(
    "How to use this figure in the paper",
    "This figure should support the sentence that the architecture is not merely sparse or over-parameterized, but deliberately decomposed across temporal scopes. In the caption or text, explicitly say that the dense and order-swap controls test credible alternative architectures, not strawman baselines."
)


## 3. Cue Compactness and Alignment Sensitivity

This figure is the cleanest place to combine the two cue-related claims:

- compact cues are already useful, and
- the model really depends on aligned cue information rather than arbitrary extra features.

A strong version has one panel for family ablation and one panel for corruption or shuffle sanity.
That combination is usually much more persuasive than a single ablation bar chart.


In [ ]:
ABLATION_PATH = RESULTS_ROOT / "04_cue_ablation/04a_cue_ablation.csv"
RETENTION_PATH = RESULTS_ROOT / "04_cue_ablation/04b_cue_retention.csv"
ABLATION_COLUMNS = ["paper_section", "panel", "dataset", "variant_or_model", "cue_setting", "cue_family_removed", "metric", "cutoff", "value", "split", "selection_rule", "run_id", "source_path", "notes"]
RETENTION_COLUMNS = ["paper_section", "panel", "dataset", "variant_or_model", "retention_target", "metric", "cutoff", "value", "reference_value", "relative_gain", "split", "selection_rule", "run_id", "source_path", "notes"]

def demo_ablation() -> pd.DataFrame:
    rows = []
    values = {
        "full": [0.0806, 0.3398],
        "memory+exposure": [0.0794, 0.3375],
        "no_exposure": [0.0788, 0.3359],
        "sequence_only": [0.0769, 0.3305],
    }
    for cue_setting, scores in values.items():
        for dataset, value in zip(["Beauty", "KuaiRec"], scores):
            rows.append({
                "paper_section": "04_cue_ablation",
                "panel": "ablation",
                "dataset": dataset,
                "variant_or_model": "RouteRec",
                "cue_setting": cue_setting,
                "cue_family_removed": cue_setting,
                "metric": "MRR",
                "cutoff": 20,
                "value": value,
                "split": "test",
                "selection_rule": "storyboard_demo",
                "run_id": "cue_demo",
                "source_path": "demo",
                "notes": "Demo cue-family plot.",
            })
    return pd.DataFrame(rows)

def demo_retention() -> pd.DataFrame:
    rows = []
    values = [
        ("Beauty", "eval_all_zero", 0.0738, 0.0806, 0.915),
        ("Beauty", "eval_all_shuffle", 0.0717, 0.0806, 0.889),
        ("Beauty", "train_global_permute", 0.0701, 0.0806, 0.870),
        ("Beauty", "role_swap", 0.0722, 0.0806, 0.896),
        ("KuaiRec", "eval_all_zero", 0.3291, 0.3398, 0.968),
        ("KuaiRec", "eval_all_shuffle", 0.3204, 0.3398, 0.943),
        ("KuaiRec", "train_global_permute", 0.3168, 0.3398, 0.932),
        ("KuaiRec", "role_swap", 0.3227, 0.3398, 0.950),
    ]
    for dataset, target, value, reference_value, relative_gain in values:
        rows.append({
            "paper_section": "04_cue_ablation",
            "panel": "retention",
            "dataset": dataset,
            "variant_or_model": "RouteRec",
            "retention_target": target,
            "metric": "MRR",
            "cutoff": 20,
            "value": value,
            "reference_value": reference_value,
            "relative_gain": relative_gain,
            "split": "test",
            "selection_rule": "storyboard_demo",
            "run_id": "cue_demo",
            "source_path": "demo",
            "notes": "Demo perturbation-retention plot.",
        })
    return pd.DataFrame(rows)

ablation_df, ablation_mode = load_csv_or_demo(ABLATION_PATH, ABLATION_COLUMNS, demo_builder=demo_ablation)
retention_df, retention_mode = load_csv_or_demo(RETENTION_PATH, RETENTION_COLUMNS, demo_builder=demo_retention)
display(Markdown(f"**Load mode:** ablation={ablation_mode}, retention={retention_mode}"))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8), constrained_layout=True)
grouped_barplot(
    ablation_df,
    x="dataset",
    hue="cue_setting",
    y="value",
    ax=axes[0],
    title="(a) Cue-family compactness",
    ylabel="MRR@20",
    xlabel="Dataset",
)
grouped_barplot(
    retention_df,
    x="dataset",
    hue="retention_target",
    y="relative_gain",
    ax=axes[1],
    title="(b) Gain retention under cue perturbation",
    ylabel="Relative gain retained",
    xlabel="Dataset",
)
saved_paths = export_figure(fig, "00_storyboard_cue_sanity", RESULTS_ROOT)
display(Markdown("Saved figures: " + ", ".join(str(path) for path in saved_paths)))
plt.show()

story_note(
    "How to use this figure in the paper",
    "The left panel supports the portability and compactness claim. The right panel is what turns the section into a causal argument: if shuffled or mismatched cues erase a meaningful fraction of the gain, then the model is relying on aligned cue semantics rather than benefiting from a generic side channel."
)


## 4. Diagnostics: Show That Routing Is Structured, Not Collapsed

These figures are usually better in the appendix, but they can still influence how convincing the whole paper feels.

The best diagnostics are not random extras. They should map back to the main claims:

- expert usage and entropy: is routing collapsing?
- top-1 fraction: is a single expert monopolizing decisions?
- feature-family consistency: do certain cue families induce stable route partitions?

This cell tries to use the actual local `test_diag.json` first. If it is missing, it falls back to demo data.


In [ ]:
DIAG_JSON = DIAG_ROOT / "raw/test_diag.json"

def load_diag_from_json(diag_path: Path):
    if not diag_path.exists():
        return None, None, None, "missing"
    payload = json.loads(diag_path.read_text())
    usage_rows = []
    summary_rows = []
    family_rows = []
    for stage_key, stage_payload in payload.get("stage_metrics", {}).items():
        stage_name = stage_key.split("@")[0]
        expert_names = stage_payload.get("expert_names", [])
        usage_sum = stage_payload.get("usage_sum", [])
        usage_total = sum(usage_sum) or 1.0
        for expert_name, usage in zip(expert_names, usage_sum):
            usage_rows.append({"stage": stage_name, "expert_label": expert_name, "usage_share": usage / usage_total})
        summary_rows.extend([
            {"stage": stage_name, "measure": "entropy_mean", "value": stage_payload.get("entropy_mean")},
            {"stage": stage_name, "measure": "route_consistency_knn_score", "value": stage_payload.get("route_consistency_knn_score")},
            {"stage": stage_name, "measure": "top1_max_frac", "value": payload.get("scalar_metrics", {}).get(f"{stage_name}_1.top1_max_frac")},
        ])
        family_payload = stage_payload.get("route_consistency_feature_group_knn", {})
        for family_name, score in zip(family_payload.get("group_names", []), family_payload.get("score_by_group", [])):
            family_rows.append({"stage": stage_name, "feature_family": family_name, "consistency_score": score})
    usage_df = pd.DataFrame(usage_rows)
    summary_df = pd.DataFrame(summary_rows).dropna()
    family_df = pd.DataFrame(family_rows)
    if usage_df.empty and summary_df.empty and family_df.empty:
        return None, None, None, "empty"
    return usage_df, summary_df, family_df, "json"

def demo_diag():
    usage = pd.DataFrame([
        {"stage": "macro", "expert_label": "Tempo_a", "usage_share": 0.11},
        {"stage": "macro", "expert_label": "Focus_a", "usage_share": 0.08},
        {"stage": "macro", "expert_label": "Memory_a", "usage_share": 0.10},
        {"stage": "macro", "expert_label": "Exposure_a", "usage_share": 0.07},
        {"stage": "mid", "expert_label": "Tempo_a", "usage_share": 0.16},
        {"stage": "mid", "expert_label": "Focus_a", "usage_share": 0.13},
        {"stage": "mid", "expert_label": "Memory_a", "usage_share": 0.14},
        {"stage": "mid", "expert_label": "Exposure_a", "usage_share": 0.10},
        {"stage": "micro", "expert_label": "Tempo_a", "usage_share": 0.21},
        {"stage": "micro", "expert_label": "Focus_a", "usage_share": 0.17},
        {"stage": "micro", "expert_label": "Memory_a", "usage_share": 0.15},
        {"stage": "micro", "expert_label": "Exposure_a", "usage_share": 0.12},
    ])
    summary = pd.DataFrame([
        {"stage": "macro", "measure": "entropy_mean", "value": 2.45},
        {"stage": "macro", "measure": "route_consistency_knn_score", "value": 0.92},
        {"stage": "macro", "measure": "top1_max_frac", "value": 0.19},
        {"stage": "mid", "measure": "entropy_mean", "value": 2.11},
        {"stage": "mid", "measure": "route_consistency_knn_score", "value": 0.88},
        {"stage": "mid", "measure": "top1_max_frac", "value": 0.24},
        {"stage": "micro", "measure": "entropy_mean", "value": 1.78},
        {"stage": "micro", "measure": "route_consistency_knn_score", "value": 0.81},
        {"stage": "micro", "measure": "top1_max_frac", "value": 0.30},
    ])
    family = pd.DataFrame([
        {"stage": "macro", "feature_family": "Tempo", "consistency_score": 0.83},
        {"stage": "macro", "feature_family": "Focus", "consistency_score": 0.87},
        {"stage": "macro", "feature_family": "Memory", "consistency_score": 0.85},
        {"stage": "macro", "feature_family": "Exposure", "consistency_score": 0.79},
        {"stage": "micro", "feature_family": "Tempo", "consistency_score": 0.71},
        {"stage": "micro", "feature_family": "Focus", "consistency_score": 0.75},
        {"stage": "micro", "feature_family": "Memory", "consistency_score": 0.74},
        {"stage": "micro", "feature_family": "Exposure", "consistency_score": 0.68},
    ])
    return usage, summary, family, "demo"

usage_df, summary_df, family_df, diag_mode = load_diag_from_json(DIAG_JSON)
if diag_mode != "json":
    usage_df, summary_df, family_df, diag_mode = demo_diag()
display(Markdown(f"**Load mode:** diagnostics={diag_mode}"))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), constrained_layout=True)
heatmap_from_long(usage_df, index="stage", columns="expert_label", values="usage_share", ax=axes[0], title="(a) Expert usage share", cmap="viridis", fmt=".2f")
grouped_barplot(summary_df, x="stage", hue="measure", y="value", ax=axes[1], title="(b) Entropy, consistency, monopoly", ylabel="Value", xlabel="Stage")
grouped_barplot(family_df, x="feature_family", hue="stage", y="consistency_score", ax=axes[2], title="(c) Family-wise routing consistency", ylabel="Consistency score", xlabel="Cue family")
saved_paths = export_figure(fig, "00_storyboard_diagnostics", RESULTS_ROOT)
display(Markdown("Saved figures: " + ", ".join(str(path) for path in saved_paths)))
plt.show()

story_note(
    "How to use this figure in the paper",
    "These plots belong naturally in the appendix, but they strengthen the credibility of the whole paper. They are useful when a reviewer asks whether the router collapsed, whether the expert partition is interpretable, or whether the consistency loss simply improves the metric without producing cleaner routing behavior."
)


## 5. PCA or Geometry View: Qualitative Only

A PCA scatter can be a nice supporting figure, but it should not carry the main claim by itself.

The safest use is:

- show that cue or router-input geometry has visible structure,
- color by stage, top-1 group, or behavior bucket,
- describe it as an intuitive visualization rather than hard causal evidence.

This cell tries to load `viz_router_input_pca.csv.gz`. If the local file is empty, it uses a small demo cloud instead.


In [ ]:
PCA_PATH = DIAG_ROOT / "tier_c_viz/viz_router_input_pca.csv.gz"

def load_pca_or_demo(pca_path: Path):
    if pca_path.exists():
        df = pd.read_csv(pca_path, compression="gzip")
        if not df.empty and {"pc1", "pc2"}.issubset(df.columns):
            mode = "csv"
            if "top1_label" not in df.columns:
                df["top1_label"] = "route"
            if "stage_name" not in df.columns:
                df["stage_name"] = "stage"
            return df, mode
    demo = pd.DataFrame([
        {"pc1": -1.8, "pc2": 1.1, "top1_label": "Tempo", "stage_name": "macro"},
        {"pc1": -1.4, "pc2": 0.9, "top1_label": "Tempo", "stage_name": "macro"},
        {"pc1": -0.7, "pc2": 0.2, "top1_label": "Focus", "stage_name": "mid"},
        {"pc1": -0.5, "pc2": 0.1, "top1_label": "Focus", "stage_name": "mid"},
        {"pc1": 0.4, "pc2": -0.3, "top1_label": "Memory", "stage_name": "mid"},
        {"pc1": 0.8, "pc2": -0.4, "top1_label": "Memory", "stage_name": "mid"},
        {"pc1": 1.5, "pc2": -1.0, "top1_label": "Exposure", "stage_name": "micro"},
        {"pc1": 1.9, "pc2": -1.2, "top1_label": "Exposure", "stage_name": "micro"},
    ])
    return demo, "demo"

pca_df, pca_mode = load_pca_or_demo(PCA_PATH)
display(Markdown(f"**Load mode:** pca={pca_mode}"))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8), constrained_layout=True)
scatterplot_with_annotations(pca_df, x="pc1", y="pc2", hue="top1_label", style="stage_name", ax=axes[0], title="(a) Router-input PCA", xlabel="PC1", ylabel="PC2")
scatterplot_with_annotations(pca_df, x="pc1", y="pc2", hue="stage_name", style="top1_label", ax=axes[1], title="(b) Same points, stage-colored", xlabel="PC1", ylabel="PC2")
saved_paths = export_figure(fig, "00_storyboard_pca", RESULTS_ROOT)
display(Markdown("Saved figures: " + ", ".join(str(path) for path in saved_paths)))
plt.show()

story_note(
    "How to use this figure in the paper",
    "Treat this as a qualitative support figure. It works best when paired with a stronger quantitative sanity result, such as cue shuffling or family-wise consistency. A good sentence is that the geometry is visually compatible with the routing story, but the main evidence comes from perturbation and consistency analyses."
)


## Suggested Paper Flow

A compact paper flow based on the figures above would be:

1. `Table`: overall ranking quality.
2. `Figure`: routing control, with both quality and consistency.
3. `Figure`: stage structure, with both removal and credible architecture alternatives.
4. `Figure`: cue compactness plus corruption sanity.
5. `Appendix figures`: diagnostics and optional PCA.

This order usually reads more convincingly than a sequence of disconnected ablation bar plots, because each figure answers a different reviewer question.

If the final results budget is tight, keep the main paper focused on:

- routing control,
- stage structure,
- cue sanity.

If more space is available, the first appendix figure to invest in is the diagnostics panel, not the PCA.
